In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
qwen_df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/qwen_labels.csv')
qwen_df = qwen_df.dropna()
label_cols = ["topic_label", "ver_label", "tax_label", "research_field"]
for col in label_cols:
    qwen_df[col] = qwen_df[col].astype(str).replace(['nan', 'None'], 'unknown')
print(qwen_df.shape)
qwen_df.head()

In [ ]:
t5_df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/testing_results_tax_ver_base_large.csv')
t5_df = t5_df.dropna()
label_cols = ["tax_large", "ver_large", "tax_base", "tax_large"]
for col in label_cols:
    t5_df[col] = t5_df[col].astype(str).replace(['nan', 'None'], 'unknown')
print(t5_df.shape)
t5_df.head()

In [ ]:
gemini_df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/test_gemini_labels.csv')
gemini_df = gemini_df.dropna()
gemini_df = gemini_df.rename(columns={'response': 'gemini_label'})
label_cols = ["gemini_label"]
for col in label_cols:
    gemini_df[col] = gemini_df[col].astype(str).replace(['nan', 'None'], 'unknown')
print(gemini_df.shape)
gemini_df.head()

In [ ]:
# 1. Define columns for the existing T5 merge and the new Gemini merge
cols_t5 = ["topic_words", "ver_base", "ver_large", "tax_base", "tax_large"]
# Assuming gemini_df also uses 'topic_words' as the key
cols_gemini = ["topic_words", "gemini_label"] 

# 2. Perform chained inner merge
df = (
    qwen_df
    .merge(t5_df[cols_t5], on='topic_words', how='inner')
    .merge(gemini_df[cols_gemini], on='topic_words', how='inner')
)

print(f"Final shape: {df.shape}")
df.head()

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_column_similarities(df, true_col, generated_cols):
    results = {}
    
    true_embeddings = model.encode(df[true_col].tolist(), convert_to_tensor=True)
    
    for gen_col in generated_cols:
        gen_embeddings = model.encode(df[gen_col].tolist(), convert_to_tensor=True)
        
        cosine_scores = util.cos_sim(true_embeddings, gen_embeddings)
        row_wise_similarities = cosine_scores.diagonal()
        mean_sim = row_wise_similarities.mean().item()
        print(f"Mean similarity for {gen_col}: {mean_sim:.4f}")
        # Store full similarity array instead of mean
        results[gen_col] = row_wise_similarities.cpu().numpy()
        
    return results

similarities = calculate_column_similarities(
    df,
    'topic_label',
    ['tax_label', 'ver_label',"ver_base", "ver_large", "tax_base", "tax_large", "gemini_label"]
)

for gen_col, scores in similarities.items():
    # This creates columns like 'tax_label_score', 'ver_label_score', etc.
    df[f"{gen_col}_score"] = scores

# 3. Verify the new columns
print(df.columns)
df.head()

In [ ]:
bins = np.arange(0, 1.01, 0.2)
labels = [f"${bins[i]:.1f}-{bins[i+1]:.1f}$" for i in range(len(bins)-1)]
x = np.arange(len(labels))

width = 0.8 / len(similarities) 

for i, (col, values) in enumerate(similarities.items()):
    counts, _ = np.histogram(values, bins=bins)
    
    offset = (i - (len(similarities) - 1) / 2) * width
    
    plt.bar(
        x + offset, 
        counts, 
        width, 
        label=col
    )

plt.xlabel("Cosine Similarity Range")
plt.ylabel("Count")
plt.title("Similarity Distribution Comparison")
plt.xticks(x, labels, rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('similarity_distribution_bars.png')

In [ ]:
label_cols = [
    'tax_label_score', 'ver_label_score', 'ver_base_score', 
    'ver_large_score', 'tax_base_score', 'tax_large_score'
]

# Calculate the signed difference for each column
for col in label_cols:
    # Creating a new column name by replacing '_score' with '_diff'
    new_col_name = col.replace('_score', '_diff')
    df[new_col_name] = df[col] - df['gemini_label_score']

# Verify the new columns
print(df.columns[df.columns.str.contains('_diff')])
df.head()

In [ ]:
diff_cols = [
    'tax_label_diff', 'ver_label_diff', 'ver_base_diff', 
    'ver_large_diff', 'tax_base_diff', 'tax_large_diff'
]

# Create a 3x2 grid of plots
fig, axes = plt.subplots(2, 3, figsize=(15, 12))
axes = axes.flatten()

auc_results = {}

for i, col in enumerate(diff_cols):
    # 1. Sort the column descendingly
    sorted_values = df[col].sort_values(ascending=False).values
    x = np.arange(len(sorted_values))
    
    # 2. Calculate Area Under Graph (Integral)
    # This represents the net performance gain/loss vs Gemini across all rows
    auc = np.trapz(sorted_values, x)
    auc_results[col] = auc
    
    # 3. Plotting
    axes[i].plot(x, sorted_values, color='steelblue', lw=2)
    axes[i].fill_between(x, sorted_values, color='skyblue', alpha=0.4)
    
    # 4. Add a reference line at zero
    axes[i].axhline(0, color='red', linestyle='--', alpha=0.6)

    axes[i].set_ylim(-0.9, 0.4)
    
    # 5. Labels and Titles
    axes[i].set_title(f"Sorted Differences: {col}")
    axes[i].set_xlabel("Ranked Row Index")
    axes[i].set_ylabel("Similarity Diff (Model - Gemini)")
    
    # Display AUC value on the plot
    axes[i].text(0.95, 0.95, f'AUC: {auc:.2f}', 
                 transform=axes[i].transAxes, 
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('similarity_diff_analysis.png')

# Print the final rankings
for col, val in sorted(auc_results.items(), key=lambda x: x[1], reverse=True):
    print(f"{col} Total AUC: {val:.4f}")